> **UFH DIRISA 2026 — Submission Notebook 2/6**  
> Project: *Eastern Cape Municipal Electoral Dynamics*  
> Scope: Eastern Cape Local Government Elections, 2000–2021 historical evidence with model-based 2026 scenarios.  
> Reproducibility rule: run notebooks in numerical order from the repository root.


# Phase 2 — Eastern Cape Demographics & Socioeconomic Features

This notebook is the **single demographic phase** for the DIRISA project.

It combines two requirements:

1. **Historical demographic alignment** for the 2011, 2016 and 2021 election backtesting period.
2. **Current/latest socioeconomic context** for the later 2026 scenario/projection stage.

## Core methodological rules

- Municipality code (`MuniCode`) is the primary geographic key.
- Census 2011 and Census 2022 municipality observations are kept as source snapshots.
- 2016 and 2021 historical features are linearly interpolated between 2011 and 2022.
- 2000 and 2006 municipality demographics remain unavailable rather than being back-extrapolated.
- Current/2022 socioeconomic variables are kept separately from historical election-year features.
- Observed, interpolated and derived variables are explicitly labelled.
- 2022 values are **not** used as if they were observed predictors for earlier elections.
- Assumption-heavy variables such as estimated voting-age population are not core model features.

This replaces the earlier Phase 2 notebook.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "data").exists() and (p / "notebooks").exists():
            return p
    raise FileNotFoundError("Run this notebook from inside the DIRISA project.")

PROJECT_ROOT = find_project_root()

MUNICIPAL_CENSUS = (
    PROJECT_ROOT / "data" / "processed" / "demographics" / "clean_census.csv"
)
PROVINCIAL_CENSUS = (
    PROJECT_ROOT / "data" / "external" / "demographics" / "EC CENSUS.csv"
)

# Optional groupmate/current socioeconomic file.
CURRENT_CANDIDATES = [
    PROJECT_ROOT / "data" / "external" / "demographics" / "EC_demographics_2022.csv",
    PROJECT_ROOT / "data" / "processed" / "demographics" / "EC_demographics_2022.csv",
    PROJECT_ROOT / "data" / "raw" / "demographics" / "EC_demographics_2022.csv",
]

PHASE2_DIR = (
    PROJECT_ROOT / "data" / "processed" / "demographics" / "phase2"
)
PHASE2_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Municipality Census:", MUNICIPAL_CENSUS.exists())
print("Provincial Census:", PROVINCIAL_CENSUS.exists())

CURRENT_FILE = next((p for p in CURRENT_CANDIDATES if p.exists()), None)
print("Current socioeconomic file:", CURRENT_FILE if CURRENT_FILE else "not present yet")


Project root: C:\Users\Monwabisi05\IdeaProjects\Dirisa_TeamQualification
Municipality Census: True
Provincial Census: True
Current socioeconomic file: not present yet


## 1. Audit demographic sources

`clean_census.csv` contains the municipality-level 2011/2022 indicators currently available in the project.

`EC CENSUS.csv` is useful as provincial Census context, but it has no municipality identifier and therefore cannot itself reconstruct the 33-row municipality table.

The optional `EC_demographics_2022.csv` is the richer current socioeconomic dataset produced by the groupmate's municipality scraper. If it is present, this notebook validates and incorporates it. If it is not present, Phase 2 still runs using the municipality Census table and reports that the current socioeconomic extension is unavailable.


In [2]:
municipal = pd.read_csv(MUNICIPAL_CENSUS)
provincial = pd.read_csv(PROVINCIAL_CENSUS)

print("Municipality Census shape:", municipal.shape)
display(municipal.head())

print("\nProvincial Census shape:", provincial.shape)
print("Provincial columns:", provincial.columns.tolist())

current = None
if CURRENT_FILE is not None:
    current = pd.read_csv(CURRENT_FILE)
    print("\nCurrent socioeconomic shape:", current.shape)
    display(current.head())


Municipality Census shape: (33, 21)


,MuniCode,Pop_2011,Pop_2022,U15_2011,U15_2022,O65_2011,O65_2022,Med_2011,Med_2022,NoSch_2011,NoSch_2022,Matric_2011,Matric_2022,Higher_2011,Higher_2022,Formal_2011,Formal_2022,Water_2011,Water_2022,Elec_2011,Elec_2022
0,BUF,781853,975255,26.5,24.8,6.1,7.8,27,30,5.1,4.1,27.1,33.6,13.2,15.0,71.9,85.6,69.1,83.7,81.1,94.6
1,EC101,79292,101001,30.2,28.3,6.8,8.1,26,28,9.3,4.2,18.4,27.4,7.6,8.0,95.3,97.2,96.9,98.1,92.1,97.3
2,EC102,36002,49883,29.2,25.0,7.0,9.2,27,31,10.5,7.8,19.0,27.2,6.1,7.7,95.9,97.2,91.4,87.4,86.9,92.2
3,EC104,80390,97815,24.4,21.9,6.2,8.2,26,31,6.3,3.8,23.0,29.6,11.7,16.6,85.4,89.6,85.2,89.7,89.5,95.2
4,EC105,61176,87797,25.2,21.3,9.9,15.1,25,35,9.7,6.3,20.4,28.2,9.6,14.6,83.6,90.9,86.0,90.4,86.3,93.4



Provincial Census shape: (576, 5)
Provincial columns: ['sex', 'count', 'census', 'age group', 'population group']


## 2. Validate municipality Census snapshots

In [3]:
EXPECTED_CODES = {
    "BUF", "NMA",
    "EC101", "EC102", "EC104", "EC105", "EC106", "EC108", "EC109",
    "EC121", "EC122", "EC123", "EC124", "EC126", "EC129",
    "EC131", "EC135", "EC136", "EC137", "EC138", "EC139",
    "EC141", "EC142", "EC145",
    "EC153", "EC154", "EC155", "EC156", "EC157",
    "EC441", "EC442", "EC443", "EC444",
}

assert len(municipal) == 33
assert municipal["MuniCode"].nunique() == 33
assert not municipal["MuniCode"].duplicated().any()
assert set(municipal["MuniCode"]) == EXPECTED_CODES
assert municipal.isna().sum().sum() == 0

pct_prefixes = [
    "U15_", "O65_", "NoSch_", "Matric_", "Higher_",
    "Formal_", "Water_", "Elec_"
]
pct_cols = [
    col for col in municipal.columns
    if any(col.startswith(prefix) for prefix in pct_prefixes)
]

for col in pct_cols:
    assert municipal[col].between(0, 100).all(), f"Invalid percentage: {col}"

assert (municipal[["Pop_2011", "Pop_2022"]] > 0).all().all()

print("PASS: 33 municipality Census snapshots validated.")


PASS: 33 municipality Census snapshots validated.


## 3. Historical demographic alignment

The historical model must not receive future Census information as though it were observed at the election date.

The alignment policy is:

| Election | Treatment |
|---|---|
| 2000 | unavailable |
| 2006 | unavailable |
| 2011 | observed Census 2011 |
| 2016 | interpolated 2011→2022 |
| 2021 | interpolated 2011→2022 |

For 2016 and 2021:

\[
x_t=x_{2011}+\frac{t-2011}{2022-2011}(x_{2022}-x_{2011})
\]

Interpolation is an analytical approximation and is labelled as such.


In [4]:
HISTORICAL_METRICS = [
    "Pop", "U15", "O65", "Med", "NoSch",
    "Matric", "Higher", "Formal", "Water", "Elec"
]
ELECTION_YEARS = [2000, 2006, 2011, 2016, 2021]

rows = []

for _, row in municipal.iterrows():
    for year in ELECTION_YEARS:
        rec = {"MuniCode": row["MuniCode"], "Year": year}

        if year < 2011:
            for metric in HISTORICAL_METRICS:
                rec[metric] = np.nan
            rec["DemographicStatus"] = "Unavailable_pre2011"
            rec["DemographicMethod"] = (
                "No municipality-level snapshot available; no back-extrapolation"
            )

        elif year == 2011:
            for metric in HISTORICAL_METRICS:
                rec[metric] = row[f"{metric}_2011"]
            rec["DemographicStatus"] = "Observed"
            rec["DemographicMethod"] = "Census 2011 municipality value"

        else:
            alpha = (year - 2011) / 11
            for metric in HISTORICAL_METRICS:
                rec[metric] = (
                    row[f"{metric}_2011"]
                    + alpha * (row[f"{metric}_2022"] - row[f"{metric}_2011"])
                )
            rec["DemographicStatus"] = "Interpolated"
            rec["DemographicMethod"] = (
                "Linear interpolation between Census 2011 and Census 2022"
            )

        rows.append(rec)

historical = (
    pd.DataFrame(rows)
    .sort_values(["MuniCode", "Year"])
    .reset_index(drop=True)
)

display(historical.head(15))


,MuniCode,Year,Pop,U15,O65,Med,NoSch,Matric,Higher,Formal,Water,Elec,DemographicStatus,DemographicMethod
0,BUF,2000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unavailable_pre2011,No municipality-level snapshot available; no b...
1,BUF,2006,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unavailable_pre2011,No municipality-level snapshot available; no b...
2,BUF,2011,781853.000000,26.500000,6.100000,27.000000,5.100000,27.100000,13.200000,71.900000,69.100000,81.100000,Observed,Census 2011 municipality value
3,BUF,2016,869763.000000,25.727273,6.872727,28.363636,4.645455,30.054545,14.018182,78.127273,75.736364,87.236364,Interpolated,Linear interpolation between Census 2011 and C...
4,BUF,2021,957673.000000,24.954545,7.645455,29.727273,4.190909,33.009091,14.836364,84.354545,82.372727,93.372727,Interpolated,Linear interpolation between Census 2011 and C...
5,EC101,2000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unavailable_pre2011,No municipality-level snapshot available; no b...
6,EC101,2006,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unavailable_pre2011,No municipality-level snapshot available; no b...
7,EC101,2011,79292.000000,30.200000,6.800000,26.000000,9.300000,18.400000,7.600000,95.300000,96.900000,92.100000,Observed,Census 2011 municipality value
8,EC101,2016,89159.727273,29.336364,7.390909,26.909091,6.981818,22.490909,7.781818,96.163636,97.445455,94.463636,Interpolated,Linear interpolation between Census 2011 and C...
9,EC101,2021,99027.454545,28.472727,7.981818,27.818182,4.663636,26.581818,7.963636,97.027273,97.990909,96.827273,Interpolated,Linear interpolation between Census 2011 and C...


## 4. Current socioeconomic feature table

The groupmate's scraper adds useful current variables such as:

- detailed age structure;
- sex ratio / female share;
- household size;
- formal, traditional and informal dwellings;
- water, electricity and toilet access;
- population density;
- municipality/rurality category.

These variables are useful for current structural description and later 2026 scenario features.

They are **not copied backwards into historical election years**.


In [5]:
def standardise_current_demographics(df):
    x = df.copy()

    # Harmonise key name.
    if "MuniCode" not in x.columns and "mdb_code" in x.columns:
        x = x.rename(columns={"mdb_code": "MuniCode"})

    if "MuniCode" not in x.columns:
        raise ValueError("Current demographic file needs MuniCode or mdb_code.")

    x["MuniCode"] = x["MuniCode"].astype(str).str.strip().str.upper()

    # Add female/male shares if only sex ratio exists.
    ratio_col = "sex_ratio_males_per_100_females"
    if ratio_col in x.columns:
        if "female_share_pct" not in x.columns:
            x["female_share_pct"] = (
                10000 / (100 + pd.to_numeric(x[ratio_col], errors="coerce"))
            ).round(2)
        if "male_share_pct" not in x.columns:
            x["male_share_pct"] = (100 - x["female_share_pct"]).round(2)

    # Keep derived service index clearly labelled.
    service_cols = [
        "piped_water_pct",
        "electricity_access_pct",
        "flush_toilet_pct",
        "formal_dwellings_pct",
    ]
    if all(col in x.columns for col in service_cols):
        if "service_delivery_index" not in x.columns:
            x["service_delivery_index"] = x[service_cols].mean(axis=1).round(2)

    # Dependency ratio is descriptive, not a voting-age measure.
    age_cols = [
        "age_0_14_pct", "age_15_34_pct",
        "age_35_59_pct", "age_60_plus_pct"
    ]
    if all(col in x.columns for col in age_cols):
        if "age_dependency_ratio" not in x.columns:
            denominator = x["age_15_34_pct"] + x["age_35_59_pct"]
            x["age_dependency_ratio"] = (
                100 * (x["age_0_14_pct"] + x["age_60_plus_pct"]) / denominator
            ).round(2)
        x["age_groups_sum_pct"] = x[age_cols].sum(axis=1).round(2)

    # Assumption-heavy estimated VAP variables are intentionally excluded
    # from the core feature list even if the source file contains them.
    return x

if current is not None:
    current_clean = standardise_current_demographics(current)

    assert not current_clean["MuniCode"].duplicated().any()
    assert set(current_clean["MuniCode"]) == EXPECTED_CODES, (
        "Current socioeconomic file does not match the 33-municipality scope."
    )

    print("PASS: current socioeconomic table matches all 33 municipality codes.")
    display(current_clean.head())
else:
    # Still produce a current table from the 2022 municipality snapshot.
    current_clean = municipal[
        ["MuniCode"] + [f"{m}_2022" for m in HISTORICAL_METRICS]
    ].copy()
    current_clean = current_clean.rename(
        columns={f"{m}_2022": m for m in HISTORICAL_METRICS}
    )
    current_clean["CurrentFeatureSource"] = (
        "clean_census.csv 2022 snapshot; richer scraper file not present"
    )
    print(
        "NOTE: EC_demographics_2022.csv not found. "
        "Using the available 2022 municipality Census snapshot."
    )


NOTE: EC_demographics_2022.csv not found. Using the available 2022 municipality Census snapshot.


## 5. Core versus optional/derived features

To avoid unnecessary feature inflation, Phase 2 preserves raw variables and does not force every constructed measure into the models.

### Core historical candidates
Population, age structure, education, formal housing, water and electricity.

### Additional current candidates
Female share, detailed age structure, household size, dwelling type, sanitation, density and rurality/category where available.

### Derived exploratory candidates
Service-delivery index and age-dependency ratio.

Estimated voting-age variables based on assumptions such as taking 85% of the 15–34 group are not included as core features.


In [6]:
historical_feature_metadata = pd.DataFrame([
    ["Pop", "Population", "Observed/interpolated", True],
    ["U15", "Population aged under 15 (%)", "Observed/interpolated", True],
    ["O65", "Population aged 65+ (%)", "Observed/interpolated", True],
    ["Med", "Median age", "Observed/interpolated", True],
    ["NoSch", "No schooling (%)", "Observed/interpolated", True],
    ["Matric", "Matric (%)", "Observed/interpolated", True],
    ["Higher", "Higher education (%)", "Observed/interpolated", True],
    ["Formal", "Formal dwelling (%)", "Observed/interpolated", True],
    ["Water", "Water access (%)", "Observed/interpolated", True],
    ["Elec", "Electricity access (%)", "Observed/interpolated", True],
], columns=["Feature", "Meaning", "Status", "CoreCandidate"])

display(historical_feature_metadata)


,Feature,Meaning,Status,CoreCandidate
0,Pop,Population,Observed/interpolated,True
1,U15,Population aged under 15 (%),Observed/interpolated,True
2,O65,Population aged 65+ (%),Observed/interpolated,True
3,Med,Median age,Observed/interpolated,True
4,NoSch,No schooling (%),Observed/interpolated,True
5,Matric,Matric (%),Observed/interpolated,True
6,Higher,Higher education (%),Observed/interpolated,True
7,Formal,Formal dwelling (%),Observed/interpolated,True
8,Water,Water access (%),Observed/interpolated,True
9,Elec,Electricity access (%),Observed/interpolated,True


## 6. Quality control

In [7]:
qc_rows = [
    ["Municipality Census rows", len(municipal), 33],
    ["Unique municipality codes", municipal["MuniCode"].nunique(), 33],
    ["Historical aligned rows", len(historical), 165],
    ["2011 observed rows", (historical["DemographicStatus"] == "Observed").sum(), 33],
    ["2016/2021 interpolated rows", (historical["DemographicStatus"] == "Interpolated").sum(), 66],
    ["2000/2006 unavailable rows", (historical["DemographicStatus"] == "Unavailable_pre2011").sum(), 66],
    ["Current municipality rows", len(current_clean), 33],
    ["Current unique municipality codes", current_clean["MuniCode"].nunique(), 33],
]

qc = pd.DataFrame(qc_rows, columns=["Check", "Actual", "Expected"])
qc["Pass"] = qc["Actual"] == qc["Expected"]

# Historical percentage checks where data are available.
available = historical[historical["Year"] >= 2011]
for col in ["U15", "O65", "NoSch", "Matric", "Higher", "Formal", "Water", "Elec"]:
    assert available[col].between(0, 100).all(), f"Invalid percentage: {col}"

# Optional current checks.
current_pct_cols = [
    "female_share_pct", "male_share_pct",
    "age_0_14_pct", "age_15_34_pct", "age_35_59_pct", "age_60_plus_pct",
    "formal_dwellings_pct", "traditional_dwellings_pct",
    "informal_dwellings_pct", "piped_water_pct",
    "electricity_access_pct", "flush_toilet_pct"
]
for col in current_pct_cols:
    if col in current_clean.columns:
        vals = pd.to_numeric(current_clean[col], errors="coerce").dropna()
        assert vals.between(0, 100).all(), f"Invalid current percentage: {col}"

display(qc)
assert qc["Pass"].all(), "Phase 2 quality-control gate failed."
print("PASS: Phase 2 quality-control checks.")


,Check,Actual,Expected,Pass
0,Municipality Census rows,33,33,True
1,Unique municipality codes,33,33,True
2,Historical aligned rows,165,165,True
3,2011 observed rows,33,33,True
4,2016/2021 interpolated rows,66,66,True
5,2000/2006 unavailable rows,66,66,True
6,Current municipality rows,33,33,True
7,Current unique municipality codes,33,33,True


PASS: Phase 2 quality-control checks.


## 7. Provenance and methodological metadata

In [8]:
provenance_rows = [
    {
        "Dataset": "EC CENSUS.csv",
        "Role": "Provincial context",
        "Geography": "Eastern Cape province",
        "Years": "1996, 2001, 2011, 2022",
        "HistoricalModelUse": "No direct municipality features",
        "Note": "No municipality identifier."
    },
    {
        "Dataset": "clean_census.csv",
        "Role": "Municipality historical demographic source currently in repository",
        "Geography": "33 harmonised municipalities",
        "Years": "2011, 2022",
        "HistoricalModelUse": "Yes",
        "Note": (
            "2011 observed; 2016/2021 interpolated. "
            "Upstream raw municipality source should be documented when available."
        )
    },
]

if CURRENT_FILE is not None:
    provenance_rows.append({
        "Dataset": CURRENT_FILE.name,
        "Role": "Current socioeconomic features",
        "Geography": "33 current Eastern Cape municipalities",
        "Years": "Primarily 2022/current",
        "HistoricalModelUse": "No",
        "Note": (
            "Used for current structural context / later scenario features; "
            "not copied into historical elections."
        )
    })

provenance = pd.DataFrame(provenance_rows)
display(provenance)


,Dataset,Role,Geography,Years,HistoricalModelUse,Note
0,EC CENSUS.csv,Provincial context,Eastern Cape province,"1996, 2001, 2011, 2022",No direct municipality features,No municipality identifier.
1,clean_census.csv,Municipality historical demographic source cur...,33 harmonised municipalities,"2011, 2022",Yes,2011 observed; 2016/2021 interpolated. Upstrea...


## 8. Save final Phase 2 outputs

In [9]:
outputs = {
    "municipality_census_snapshots_2011_2022.csv": municipal,
    "demographics_historical_aligned_2000_2021.csv": historical,
    "demographics_current_2022.csv": current_clean,
    "phase2_feature_metadata.csv": historical_feature_metadata,
    "phase2_provenance.csv": provenance,
    "phase2_quality_control.csv": qc,
}

for filename, df in outputs.items():
    path = PHASE2_DIR / filename
    df.to_csv(path, index=False)
    print("Saved:", path.relative_to(PROJECT_ROOT), "|", len(df), "rows")

print("\nPhase 2 complete.")


Saved: data\processed\demographics\phase2\municipality_census_snapshots_2011_2022.csv | 33 rows
Saved: data\processed\demographics\phase2\demographics_historical_aligned_2000_2021.csv | 165 rows
Saved: data\processed\demographics\phase2\demographics_current_2022.csv | 33 rows
Saved: data\processed\demographics\phase2\phase2_feature_metadata.csv | 10 rows
Saved: data\processed\demographics\phase2\phase2_provenance.csv | 2 rows
Saved: data\processed\demographics\phase2\phase2_quality_control.csv | 8 rows

Phase 2 complete.


## Phase 2 completion note

Phase 2 now handles the entire demographic side of the project in one place.

### Historical modelling
Use `demographics_historical_aligned_2000_2021.csv`.

- 2011 = observed
- 2016/2021 = interpolated
- 2000/2006 = unavailable

### Current/latest structural context
Use `demographics_current_2022.csv`.

This may contain the richer scraper variables if `EC_demographics_2022.csv` is present. It must not be treated as historical data.

### Phase 3
Phase 3 will merge the historical demographic table with the Phase 1 municipality-election panel using:

`MuniCode + Year`

The five-election panel remains useful for electoral trend EDA, while the demographic modelling subset begins in 2011.


## Submission interpretation

Phase 2 separates observed demographic evidence from derived historical alignment. Municipality-level 2011 and 2022 Census snapshots are retained as anchors; 2016 and 2021 values are linearly interpolated for descriptive historical analysis, while pre-2011 municipality demographics remain unavailable rather than being back-extrapolated.

**Leakage rule:** 2022-informed interpolated variables are not treated as pre-2021 predictors in the strict Phase 4 historical forecast. Current 2022 demographics are used as contextual information for the 2026 scenario layer.